In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('proact_preprocessed_S1.csv')
df_mcmc = pd.read_csv("../raw_results/results_S1_dahlou_ncp_cov_run1.csv")

In [3]:
import pandas as pd
import numpy as np
import re
from scipy.special import expit
from sklearn.metrics import mean_squared_error

# ==========================================
# 0. PREPARATION & DIMENSIONS
# ==========================================
# IMPORTANT: Ensure your longitudinal data is loaded as 'df'
# df = pd.read_csv("your_longitudinal_data.csv")

# Load the MCMC summary

# Dynamically calculate dimensions from your dataframe
N = len(df)                          # Total observations (4336)
Nsub = df['subject_id'].nunique()    # Number of unique subjects (775)
K = 12                               # Number of ALSFRS-R items
R = 4                                # Number of latent domains
p_meas = 3                           # Number of measurement covariates

# ==========================================
# 1. RECONSTRUCT DATA ARRAYS FROM 'df'
# ==========================================
print("Reconstructing data arrays...")

# 1. ID array (Stan requires 1-indexed contiguous integers)
subject_mapping = {sub: i+1 for i, sub in enumerate(df['subject_id'].unique())}
ID = df['subject_id'].map(subject_mapping).values

# 2. X_meas Matrix (Covariates)
# Ensure these names exactly match the columns in your 'df'
X_meas = df[['Sex_Female', 'Treatment_Active', 'Age_Base']].values

# 3. Y Matrix (Observed Items)
all_items = [
    'Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing', 
    'Q4_Handwriting', 'Q5_Cutting', 'Q6_Dressing_and_Hygiene', 
    'Q7_Turning_in_Bed', 'Q8_Walking', 'Q9_Climbing_Stairs', 
    'R_1_Dyspnea', 'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency'
]
# We extract the raw 0-4 scores directly for comparison
actual_scores = df[all_items].values

# ==========================================
# 2. EXTRACT POSTERIOR MEANS
# ==========================================
print("Extracting posterior means from MCMC results...")

def extract_stan_param(df_summary, param_name, shape):
    """Extracts 'Estimate' column and reshapes it into a 0-indexed Numpy array."""
    out = np.zeros(shape)
    pattern = rf"^{param_name}\["
    param_df = df_summary[df_summary['Parameter'].str.contains(pattern, regex=True)]
    
    for _, row in param_df.iterrows():
        name = row['Parameter']
        val = row['Estimate']
        
        # Extract indices inside the brackets: e.g., "theta[1,2]" -> "1,2"
        idx_str = re.search(r'\[(.*?)\]', name).group(1)
        indices = [int(i) - 1 for i in idx_str.split(',')] # Convert to 0-index
        
        if len(indices) == 1:
            out[indices[0]] = val
        elif len(indices) == 2:
            out[indices[0], indices[1]] = val
        elif len(indices) == 3:
            out[indices[0], indices[1], indices[2]] = val
            
    return out

# Extract parameters into numpy arrays
theta = extract_stan_param(df_mcmc, 'theta', (K, 4))
lam   = extract_stan_param(df_mcmc, 'lambda', (K,))
beta  = extract_stan_param(df_mcmc, 'beta', (K, p_meas))
xi    = extract_stan_param(df_mcmc, 'xi', (R, N))
b     = extract_stan_param(df_mcmc, 'b', (Nsub, K))

# ==========================================
# 3. CALCULATE EXPECTED SCORES
# ==========================================
print("Calculating expected scores...")

domain_map = {
    0: [0, 1, 2],    # Bulbar
    1: [3, 4, 5],    # Fine Motor
    2: [6, 7, 8],    # Gross Motor
    3: [9, 10, 11]   # Respiratory
}

def expected_ordered_logistic(eta, cutpoints):
    """Calculates expected score (0-4) from ordered logistic parameters."""
    p_le_1 = expit(cutpoints[0] - eta)
    p_le_2 = expit(cutpoints[1] - eta)
    p_le_3 = expit(cutpoints[2] - eta)
    p_le_4 = expit(cutpoints[3] - eta)
    
    p0 = p_le_1
    p1 = p_le_2 - p_le_1
    p2 = p_le_3 - p_le_2
    p3 = p_le_4 - p_le_3
    p4 = 1.0 - p_le_4
    
    return (0*p0) + (1*p1) + (2*p2) + (3*p3) + (4*p4)

expected_scores = np.zeros((N, K))

# Loop through all observations and calculate expected 0-4 item scores
for i in range(N):
    sub_idx = ID[i] - 1  # 0-indexed subject
    xi_m = X_meas[i, :]  # (p_meas,) array
    
    for d, items in domain_map.items():
        for k in items:
            # eta = Xi_m * beta + lambda * xi + b
            eta = np.dot(xi_m, beta[k]) + lam[k] * xi[d, i] + b[sub_idx, k]
            
            # Get expected score on the 0-4 scale
            expected_scores[i, k] = expected_ordered_logistic(eta, theta[k])

# ==========================================
# 4. COMPUTE RMSE AND FORMAT OUTPUT
# ==========================================
print("Computing final metrics...\n")
results_list = []

# Individual Item RMSE
for k in range(K):
    rmse = np.sqrt(mean_squared_error(actual_scores[:, k], expected_scores[:, k]))
    results_list.append({'Level': 'Individual Item', 'Target': all_items[k], 'In-Sample RMSE': rmse})

# Domain RMSE
domain_names = {0: 'Bulbar', 1: 'Fine_Motor', 2: 'Gross_Motor', 3: 'Respiratory'}
for d, items in domain_map.items():
    expected_domain = expected_scores[:, items].sum(axis=1)
    actual_domain = actual_scores[:, items].sum(axis=1)
    rmse = np.sqrt(mean_squared_error(actual_domain, expected_domain))
    results_list.append({'Level': 'Domain', 'Target': domain_names[d], 'In-Sample RMSE': rmse})

# Total Score RMSE
expected_total = expected_scores.sum(axis=1)
actual_total = actual_scores.sum(axis=1)
rmse_total = np.sqrt(mean_squared_error(actual_total, expected_total))
results_list.append({'Level': 'Overall', 'Target': 'Total Score', 'In-Sample RMSE': rmse_total})

# Generate formatted table
df_eval = pd.DataFrame(results_list)
df_eval['Level'] = pd.Categorical(df_eval['Level'], categories=['Individual Item', 'Domain', 'Overall'], ordered=True)
df_eval = df_eval.sort_values(['Level', 'Target'])

print("="*80)
print("IRT + CONTINUOUS-TIME OU PROCESS: IN-SAMPLE PREDICTIVE ACCURACY")
print("="*80)
print(df_eval.to_string(formatters={'In-Sample RMSE': '{:.3f}'.format}, index=False))

Reconstructing data arrays...
Extracting posterior means from MCMC results...
Calculating expected scores...
Computing final metrics...

IRT + CONTINUOUS-TIME OU PROCESS: IN-SAMPLE PREDICTIVE ACCURACY
          Level                        Target In-Sample RMSE
Individual Item                     Q1_Speech          0.362
Individual Item                 Q2_Salivation          0.390
Individual Item                 Q3_Swallowing          0.345
Individual Item                Q4_Handwriting          0.387
Individual Item                    Q5_Cutting          0.296
Individual Item       Q6_Dressing_and_Hygiene          0.350
Individual Item             Q7_Turning_in_Bed          0.399
Individual Item                    Q8_Walking          0.384
Individual Item            Q9_Climbing_Stairs          0.618
Individual Item                   R_1_Dyspnea          0.435
Individual Item                 R_2_Orthopnea          0.277
Individual Item R_3_Respiratory_Insufficiency          0.153
      

In [4]:
import pandas as pd
import numpy as np
import re
from scipy.special import expit
from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. LOAD DATA & DEFINE CONSTANTS
# ==========================================
print("Loading data...")

# IMPORTANT: Update this string with the actual filename of your patient dataframe
# df = pd.read_csv("your_longitudinal_data.csv") 

# Load the MCMC summary
df = pd.read_csv('proact_preprocessed_S1.csv')
df_mcmc = pd.read_csv("../raw_results/results_S1_dahlou_ncp_cov_run1.csv")

# Dimensions
N = len(df)                          # Total observations
Nsub = df['subject_id'].nunique()    # Number of unique subjects
K = 12                               # Number of ALSFRS-R items
R = 4                                # Number of latent domains
p_meas = 3                           # Number of measurement covariates

# ==========================================
# 1. RECONSTRUCT DATA ARRAYS FROM 'df'
# ==========================================
print("Reconstructing data arrays...")

# 1. ID array (Stan requires 1-indexed contiguous integers)
subject_mapping = {sub: i+1 for i, sub in enumerate(df['subject_id'].unique())}
ID = df['subject_id'].map(subject_mapping).values

# 2. X_meas Matrix (Covariates: Gender, Treatment, Age)
X_meas = df[['Sex_Female', 'Treatment_Active', 'Age_Base']].values

# 3. Y Matrix (Observed Items on the 0-4 scale)
all_items = [
    'Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing', 
    'Q4_Handwriting', 'Q5_Cutting', 'Q6_Dressing_and_Hygiene', 
    'Q7_Turning_in_Bed', 'Q8_Walking', 'Q9_Climbing_Stairs', 
    'R_1_Dyspnea', 'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency'
]
actual_scores = df[all_items].values

# ==========================================
# 2. EXTRACT POSTERIOR MEANS
# ==========================================
print("Extracting posterior means from MCMC results...")

def extract_stan_param(df_summary, param_name, shape):
    """Extracts 'Estimate' column and reshapes it into a 0-indexed Numpy array."""
    out = np.zeros(shape)
    pattern = rf"^{param_name}\["
    param_df = df_summary[df_summary['Parameter'].str.contains(pattern, regex=True)]
    
    for _, row in param_df.iterrows():
        name = row['Parameter']
        val = row['Estimate']
        
        # Extract indices inside the brackets and convert to 0-index
        idx_str = re.search(r'\[(.*?)\]', name).group(1)
        indices = [int(i) - 1 for i in idx_str.split(',')] 
        
        if len(indices) == 1:
            out[indices[0]] = val
        elif len(indices) == 2:
            out[indices[0], indices[1]] = val
        elif len(indices) == 3:
            out[indices[0], indices[1], indices[2]] = val
            
    return out

theta = extract_stan_param(df_mcmc, 'theta', (K, 4))
lam   = extract_stan_param(df_mcmc, 'lambda', (K,))
beta  = extract_stan_param(df_mcmc, 'beta', (K, p_meas))
xi    = extract_stan_param(df_mcmc, 'xi', (R, N))
b     = extract_stan_param(df_mcmc, 'b', (Nsub, K))

# ==========================================
# 3. CALCULATE EXPECTED SCORES
# ==========================================
print("Calculating expected scores...")

domain_map = {
    0: [0, 1, 2],    # Bulbar
    1: [3, 4, 5],    # Fine Motor
    2: [6, 7, 8],    # Gross Motor
    3: [9, 10, 11]   # Respiratory
}

def expected_ordered_logistic(eta, cutpoints):
    """Calculates expected score (0-4) from ordered logistic parameters."""
    p_le_1 = expit(cutpoints[0] - eta)
    p_le_2 = expit(cutpoints[1] - eta)
    p_le_3 = expit(cutpoints[2] - eta)
    p_le_4 = expit(cutpoints[3] - eta)
    
    p0 = p_le_1
    p1 = p_le_2 - p_le_1
    p2 = p_le_3 - p_le_2
    p3 = p_le_4 - p_le_3
    p4 = 1.0 - p_le_4
    
    return (0*p0) + (1*p1) + (2*p2) + (3*p3) + (4*p4)

expected_scores = np.zeros((N, K))

# Loop through all observations
for i in range(N):
    sub_idx = ID[i] - 1  
    xi_m = X_meas[i, :]  
    
    for d, items in domain_map.items():
        for k in items:
            # eta = Xi_m * beta + lambda * xi + b
            eta = np.dot(xi_m, beta[k]) + lam[k] * xi[d, i] + b[sub_idx, k]
            expected_scores[i, k] = expected_ordered_logistic(eta, theta[k])


# ==========================================
# 4. COMPUTE ADVANCED METRICS
# ==========================================
print("Computing final advanced metrics...\n")
results_list = []

# --- Individual Item Metrics (0-4 Scale) ---
for k in range(K):
    actual_array = actual_scores[:, k]
    expected_array = expected_scores[:, k]
    
    rmse = np.sqrt(mean_squared_error(actual_array, expected_array))
    mae = mean_absolute_error(actual_array, expected_array)
    
    # Categorical classification metrics
    predicted_categories = np.clip(np.round(expected_array), 0, 4)
    exact_match = np.mean(predicted_categories == actual_array)
    kappa = cohen_kappa_score(actual_array, predicted_categories, weights='linear')
    
    results_list.append({
        'Level': 'Individual Item', 
        'Target': all_items[k], 
        'RMSE': rmse, 
        'MAE': mae,
        'Exact_Match_%': exact_match * 100,
        'Weighted_Kappa': kappa
    })

# --- Domain Metrics (0-12 Scale) ---
domain_names = {0: 'Bulbar', 1: 'Fine_Motor', 2: 'Gross_Motor', 3: 'Respiratory'}
for d, items in domain_map.items():
    actual_domain = actual_scores[:, items].sum(axis=1)
    expected_domain = expected_scores[:, items].sum(axis=1)
    
    rmse = np.sqrt(mean_squared_error(actual_domain, expected_domain))
    mae = mean_absolute_error(actual_domain, expected_domain)
    
    results_list.append({
        'Level': 'Domain', 
        'Target': domain_names[d], 
        'RMSE': rmse, 
        'MAE': mae,
        'Exact_Match_%': np.nan,
        'Weighted_Kappa': np.nan
    })

# --- Total Score Metrics (0-48 Scale) ---
actual_total = actual_scores.sum(axis=1)
expected_total = expected_scores.sum(axis=1)

rmse_total = np.sqrt(mean_squared_error(actual_total, expected_total))
mae_total = mean_absolute_error(actual_total, expected_total)

results_list.append({
    'Level': 'Overall', 
    'Target': 'Total Score', 
    'RMSE': rmse_total, 
    'MAE': mae_total,
    'Exact_Match_%': np.nan, 
    'Weighted_Kappa': np.nan
})

# ==========================================
# 5. FORMAT AND PRINT TABLE
# ==========================================
df_eval = pd.DataFrame(results_list)
df_eval['Level'] = pd.Categorical(df_eval['Level'], categories=['Overall', 'Domain', 'Individual Item'], ordered=True)
df_eval = df_eval.sort_values(['Level', 'Target'])

print("="*95)
print("IRT + CONTINUOUS-TIME OU PROCESS: ADVANCED PREDICTIVE METRICS")
print("="*95)

formatters = {
    'RMSE': '{:.3f}'.format,
    'MAE': '{:.3f}'.format,
    'Exact_Match_%': '{:.1f}%'.format,
    'Weighted_Kappa': '{:.3f}'.format
}
print(df_eval.to_string(formatters=formatters, index=False, na_rep='-'))

Loading data...
Reconstructing data arrays...
Extracting posterior means from MCMC results...
Calculating expected scores...
Computing final advanced metrics...

IRT + CONTINUOUS-TIME OU PROCESS: ADVANCED PREDICTIVE METRICS
          Level                        Target  RMSE   MAE Exact_Match_% Weighted_Kappa
        Overall                   Total Score 1.410 1.079             -              -
         Domain                        Bulbar 0.685 0.442             -              -
         Domain                    Fine_Motor 0.502 0.358             -              -
         Domain                   Gross_Motor 0.974 0.731             -              -
         Domain                   Respiratory 0.490 0.282             -              -
Individual Item                     Q1_Speech 0.362 0.205         83.3%          0.854
Individual Item                 Q2_Salivation 0.390 0.228         83.3%          0.797
Individual Item                 Q3_Swallowing 0.345 0.189         86.7%         